## PDF extraction

In [1]:
import contextlib
from collections.abc import Iterable
from pathlib import Path

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import (
    EasyOcrOptions,
    HeadingHierarchyOptions,
    PdfPipelineOptions,
    PictureDescriptionVlmOptions,
    TableFormerMode,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode


class Pdf2DoclingDocument:
    """Convert native PDFs to DoclingDocuments (headings, tables, captioned pictures, OCR).

    Unlike jpg files, where we build the DoclingDocument ourselves, PDFs go through Docling's
    automatic pipeline. A PDF page mixes text, headings, tables and images, and rebuilding the
    layout analysis, reading order, table structure and picture extraction by hand would be
    far more complex than configuring the existing pipeline.

    Known limitations observed on our corpus:
      - Picture classification and description are not reliable: some images are not
        classified or described, and descriptions can be inaccurate.
      - Table extraction has issues: some tables come out incomplete or badly structured,
        so important tables should be checked against the original PDF.
      - OCR does not seem to extract the text inside images. In the default OCR mode, OCR
        runs on regions found by the layout model that lack native PDF text, so text in an
        image that is not detected as its own region, or that overlaps native text, may be
        missed.
    """

    DEFAULT_IMAGE_PROMPT = (
        "Décris cette image extraite d'un document français, pour qu'une personne qui ne la voit pas "
        "comprenne son contenu. Réponds uniquement en français, sans introduction, sans rien inventer.\n\n"
        "Réponds exactement dans ce format Markdown, sans titre (#) :\n\n"
        "**Type :** schéma, organigramme, graphique, tableau, logo, icône, capture d'écran, "
        "palette de couleurs, photographie ou autre\n\n"
        "**Description :** selon le type :\n"
        "- schéma ou organigramme : une liste à puces, un élément ou un lien par ligne (« A → B »), "
        "dans l'ordre des étapes ;\n"
        "- graphique : le type, les axes, les séries, les valeurs lisibles et la tendance ;\n"
        "- tableau : un tableau Markdown reprenant les colonnes et les lignes ;\n"
        "- palette de couleurs : un tableau Markdown | Couleur | Code |, une ligne par couleur ;\n"
        "- logo ou icône : sa forme et ses couleurs, en une ou deux phrases ;\n"
        "- capture d'écran ou maquette : une liste à puces des zones visibles et de leur contenu ;\n"
        "- photographie : les personnes, objets, lieu, action et ambiance, en quelques phrases.\n\n"
        "**Texte visible :** tout le texte lisible dans l'image, recopié exactement dans l'ordre de lecture, "
        "une ligne par bloc de texte. Écris « Aucun » seulement s'il n'y a vraiment aucun texte dans l'image."
    )

    def __init__(
        self,
        artifacts_path: str | Path,
        vlm_model_path: str | Path,
        image_prompt: str | None = None,
        ocr_langs: list[str] | None = None,
        num_threads: int = 4,
    ) -> None:
        # Docling loads the description model from "artifacts_path / repo_id":
        # link the local VLM there so the other Docling models stay in artifacts_path
        vlm_model_path = Path(vlm_model_path)
        link = Path(artifacts_path) / vlm_model_path.name
        if not link.exists():
            link.symlink_to(vlm_model_path, target_is_directory=True)

        opts = PdfPipelineOptions(artifacts_path=artifacts_path)

        # recover PDF heading levels
        # source: https://docling-project.github.io/docling/usage/advanced_options/#recover-pdf-heading-levels
        opts.heading_hierarchy_options = HeadingHierarchyOptions(enabled=True)
        opts.generate_parsed_pages = True  # required by the font-style signal

        # rebuild tables as real rows and columns instead of loose text
        # source: https://docling-project.github.io/docling/reference/pipeline_options/#docling.datamodel.pipeline_options.TableStructureOptions.do_cell_matching
        opts.do_table_structure = True
        opts.table_structure_options = TableStructureOptions(do_cell_matching=True)
        opts.table_structure_options.mode = TableFormerMode.ACCURATE  # more accurate TableFormer model

        # extract images from the pdf, describe them for the retrieval system and the LLM
        # source: https://docling-project.github.io/docling/_generated/examples/export_figures/
        # source: https://docling-project.github.io/docling/usage/enrichments/#picture-classification
        opts.generate_picture_images = True
        opts.images_scale = 2
        opts.do_picture_classification = False  # classification with the picture description model
        opts.do_picture_description = True
        opts.picture_description_options = PictureDescriptionVlmOptions(
            repo_id="Qwen3-VL-32B-Instruct",
            prompt=image_prompt or self.DEFAULT_IMAGE_PROMPT,
        )

        # TODO: check if ocr is working
        # read text that exists only as pixels (labels inside diagrams, scanned inserts)
        # source: https://docling-project.github.io/docling/_generated/examples/custom_convert/
        opts.do_ocr = True
        opts.ocr_options = EasyOcrOptions(lang=ocr_langs or ["fr"])

        # use the GPU if available, otherwise CPU threads
        # source: https://docling-project.github.io/docling/_generated/examples/custom_convert/
        opts.accelerator_options = AcceleratorOptions(num_threads=num_threads, device=AcceleratorDevice.AUTO)

        self.converter = DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
        )

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            name = Path(result.input.file).stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            with contextlib.chdir(out_dir):  # keeps the image folder next to the files, not nested
                doc.save_as_json(Path(f"{name}.json"), image_mode=ImageRefMode.REFERENCED)
                doc.save_as_markdown(Path(f"{name}.md"), image_mode=ImageRefMode.REFERENCED)
            print("SAVED:", name, "|", len(doc.tables), "tables |", len(doc.pictures), "pictures")

In [2]:
pdf_paths = [
    "data/INJA/Livrable 1/Donnée entrée/Charte graphique INJA/CHARTE_GRAPHIQUE_INJA.pdf",
    "data/INJA/Livrable 1/Donnée entrée/DGCS commentaire relecture Version 1/20260331 Analyse livrable 1 par DGCS.pdf",
    "data/INJA/Livrable 1/Donnée entrée/Estimation couts/BP ArtSign Tradlitteraire .pdf",
    "data/INJA/Livrable 1/Donnée entrée/Organisme de médiation/Mission médiation.pdf",
]

pdf_converter = Pdf2DoclingDocument(artifacts_path="/linkhome/idris/genini01/ssos027/.cache/docling/models",
                                    vlm_model_path="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3-VL-32B-Instruct")

# 1. Convert PDF
out_dir = Path("data/INJA/Livrable 1/converted/pdf2")
pdf_converter.convert_all(pdf_paths, out_dir)
#converted_doc = pdf_converter.converter.convert(pdf_paths[0]).document

Loading weights:   0%|          | 0/1058 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-10-26 11:30:42,920 MatchingPostProcessor WARNING  161 of 162 pdf cells matched neither a row nor a column band of the 2x5 grid and were dropped from the table


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SAVED: CHARTE_GRAPHIQUE_INJA | 4 tables | 60 pictures
SAVED: 20260331 Analyse livrable 1 par DGCS | 4 tables | 0 pictures
SAVED: BP ArtSign Tradlitteraire | 3 tables | 0 pictures
SAVED: Mission médiation | 1 tables | 12 pictures


In [ ]:
from docling_core.types.doc import DoclingDocument

imported_doc = DoclingDocument.load_from_json(Path("data/INJA/Livrable 1/converted/pdf2/Mission médiation.json"))
print(imported_doc.export_to_markdown())

In [3]:
from dataclasses import dataclass
from pathlib import Path

from transformers import AutoTokenizer
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.types.doc import DoclingDocument


@dataclass
class LeafInfo:
    document: str
    headings: list[str]
    pages: list[int]

def prepare_leaves(json_dir: Path, chunker: HybridChunker, min_words: int = 20) -> tuple[list[str], list[str], list[LeafInfo]]:
    """Chunk every DoclingDocument JSON into three aligned lists: texts, cluster_texts and sources."""
    texts: list[str] = []          # document title + headings + content -> LLM summaries and search index
    cluster_texts: list[str] = []  # headings + content -> clustering (no document title)
    infos: list[LeafInfo] = []     # sources
    seen: set[str] = set()
    for json_path in sorted(json_dir.glob("*.json")):
        doc = DoclingDocument.load_from_json(json_path)
        doc_title = doc.name
        for chunk in chunker.chunk(dl_doc=doc):
            if len(chunk.text.split()) < min_words or chunk.text in seen:
                continue  # noise or exact duplicate
            seen.add(chunk.text)
            context = chunker.contextualize(chunk=chunk)
            texts.append(f"{doc_title}\n{context}")
            cluster_texts.append(context)
            pages = sorted({p.page_no for item in chunk.meta.doc_items for p in item.prov})
            infos.append(LeafInfo(doc.name, chunk.meta.headings or [], pages))
    return texts, cluster_texts, infos

# Chunker: embedding-model tokenizer
tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained("/linkhome/idris/genini01/ssos027/.cache/docling/models/all-MiniLM-L6-v2"),
    max_tokens=220,  # 256 minus a margin for the document title and headings
)
chunker = HybridChunker(tokenizer=tokenizer)
texts, cluster_texts, infos = prepare_leaves(Path("data/INJA/Livrable 1/converted/pdf2"), chunker)

print(len(texts), "feuilles")
for i in range(20,30):
    print(f"=== {i} === {infos[i]}\n{texts[i]}\n")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3184 > 512). Running this sequence through the model will result in indexing errors


156 feuilles
=== 20 === LeafInfo(document='20260331 Analyse livrable 1 par DGCS', headings=['2 Analyse globale'], pages=[5])
20260331 Analyse livrable 1 par DGCS
2 Analyse globale
-  L'utilisation, sans toujours de distinction, des termes « marché », « AMI » et « AAP » 1 , de CCTP et de cahier des charges, pour évoquer la procédure de sélection des organismes adaptateurs et les commandes ad hoc.
-  Une utilisation indifférenciée des termes « handicap » et « handicap de lecture » 2 pour l'empêchement d'accéder à la lecture (considération juridique dépassant la simple sémantique) ;
-  La proposition « d'exclure la communauté sourde » du plan d'adaptation, alors que ce sont les adaptations en LSF ou LfPC que l'INJA propose d'exclure dans un premier temps, pour des raisons de coûts, et pas une catégorie de bénéficiaires.

=== 21 === LeafInfo(document='20260331 Analyse livrable 1 par DGCS', headings=['3 Le plan de production', '3.1 Temporalité'], pages=[5])
20260331 Analyse livrable 1 pa

In [4]:
import json
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import torch
import umap
from sentence_transformers import SentenceTransformer
from sklearn.mixture import GaussianMixture
from transformers import AutoModelForImageTextToText, AutoProcessor


@dataclass
class Node:
    text: str
    level: int  # 0 = original chunk, 1+ = summary
    children: list[int] = field(default_factory=list)

class Raptor:
    """RAPTOR (Sarthi et al., 2024), adapted from the LangChain cookbook to local models.

    Source: https://github.com/langchain-ai/langchain/blob/v0.1.17rc1/cookbook/RAPTOR.ipynb

    Builds a tree bottom-up: embed texts, reduce them with UMAP, cluster them with a Gaussian mixture
    (global then local, soft membership), summarize each cluster with an LLM, and repeat on the summaries.
    Retrieval searches all nodes at once ("collapsed tree").
    """

    SUMMARY_PROMPT = (
        "Voici des extraits de documents. Rédige en français un résumé fidèle et dense qui conserve "
        "les faits, chiffres, noms, décisions et documents cités. N'invente rien, sans introduction.\n\n{texts}"
    )

    def __init__(self, embed_model: str | Path, llm_model: str | Path,
                 reduced_dim: int = 10, threshold: float = 0.1, max_clusters: int = 50, local_neighbors: int = 10,
                 n_levels: int = 3, max_new_tokens: int = 220, random_state: int = 224) -> None:
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # use config
        self.embedder = SentenceTransformer(str(embed_model), device=self.device)
        self.processor = AutoProcessor.from_pretrained(llm_model)
        self.llm = AutoModelForImageTextToText.from_pretrained(llm_model, dtype="auto").to(self.device)
        self.reduced_dim = reduced_dim
        self.threshold = threshold
        self.max_clusters = max_clusters
        self.local_neighbors = local_neighbors
        self.n_levels = n_levels
        self.max_new_tokens = max_new_tokens
        self.random_state = random_state

        # output storage
        self.nodes: list[Node] = []
        self.embeddings = np.empty((0, 0))

    def embed(self, texts: list[str]) -> np.ndarray:
        return self.embedder.encode(texts, normalize_embeddings=True)

    def _bic(self, reduced: np.ndarray, k: int) -> float:
        """BIC of a k-component Gaussian mixture fitted on the reduced embeddings (lower is better)."""
        return GaussianMixture(k, random_state=self.random_state).fit(reduced).bic(reduced)

    def gmm_cluster(self, reduced: np.ndarray) -> list[list[int]]:
        """Pick the cluster count by BIC, then keep every text whose probability exceeds the threshold.
        
        Example with 4 texts, best_k = 3 and threshold = 0.1:
            probs (texts × clusters)        output
            text 0: [0.95, 0.04, 0.01]      [[0, 2],   # cluster 0: texts 0 and 2
            text 1: [0.02, 0.97, 0.01]       [1, 2],   # cluster 1: texts 1 and 2
            text 2: [0.60, 0.35, 0.05]       [3]]      # cluster 2: text 3
            text 3: [0.01, 0.02, 0.97]
        """
        bics = [self._bic(reduced, k) for k in range(1, min(self.max_clusters, len(reduced)))]
        best_k = int(np.argmin(bics)) + 1  # bics[0] is k=1
        probs = GaussianMixture(best_k, random_state=self.random_state).fit(reduced).predict_proba(reduced)
        return [np.flatnonzero(probs[:, c] > self.threshold).tolist() for c in range(best_k)]

    def _reduce(self, embeddings: np.ndarray, n_neighbors: int) -> np.ndarray:
        n_neighbors = max(2, min(n_neighbors, len(embeddings) - 1))  # UMAP needs 2 <= n_neighbors < n
        return umap.UMAP(n_neighbors=n_neighbors, n_components=self.reduced_dim, metric="cosine",
                         random_state=self.random_state).fit_transform(embeddings)

    def cluster(self, embeddings: np.ndarray) -> list[list[int]]:
        """Global clustering, then local clustering inside each global cluster. Returns lists of indices."""
        if len(embeddings) <= self.reduced_dim + 1:
            return [list(range(len(embeddings)))]  # too few texts for UMAP dim reduction

        clusters = []
        for members in self.gmm_cluster(self._reduce(embeddings, int((len(embeddings) - 1) ** 0.5))):
            if len(members) == 0:
                continue
            if len(members) <= self.reduced_dim + 1:
                clusters.append(members)  # keep the small global cluster as it is
                continue
            local_clusters = self.gmm_cluster(self._reduce(embeddings[members], self.local_neighbors))
            for local_positions in local_clusters:
                if len(local_positions) == 0:
                    continue  # empty local cluster
                # map local positions (within this global cluster) to global positions in `embeddings`
                clusters.append([members[p] for p in local_positions])
        return clusters
        
    def summarize(self, texts: list[str]) -> str:
        prompt = self.SUMMARY_PROMPT.format(texts="\n\n---\n\n".join(texts))
        chat = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        text = self.processor.apply_chat_template(chat, add_generation_prompt=True)
        inputs = self.processor(text=text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            # for summaries we want faithful, reproducible text, not variety
            out = self.llm.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False)
        return self.processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def build(self, texts: list[str], cluster_texts: list[str] | None = None) -> None:
        """texts: read by the LLM and indexed; cluster_texts: used to cluster the leaves (default: texts)."""
        self.nodes = [Node(text=t, level=0) for t in texts]
    
        curr_layer_node_idx = list(range(len(self.nodes)))
        for level in range(1, self.n_levels + 1):
            # Clusterize the current layer
            to_embed = cluster_texts if (cluster_texts and level == 1) else [self.nodes[i].text for i in curr_layer_node_idx]
            clusters = self.cluster(self.embed(to_embed))
            print(f"level {level}: {len(curr_layer_node_idx)} nodes -> {len(clusters)} clusters "
                  f"(sizes: {sorted(len(c) for c in clusters)})")

            # Build the next layer from clusters
            next_layer_node_idx = []
            for members in clusters:
                children_idx = [curr_layer_node_idx[j] for j in members] 
                summary = self.summarize([self.nodes[i].text for i in children_idx])
                self.nodes.append(Node(text=summary, level=level, children=children_idx))
                next_layer_node_idx.append(len(self.nodes) - 1)
            curr_layer_node_idx = next_layer_node_idx
            if len(clusters) == 1:
                break  # a single cluster: its summary is the root
        self.embeddings = self.embed([n.text for n in self.nodes])

    def search(self, query: str, k: int = 5) -> list[tuple[int, float]]:
        """Collapsed-tree retrieval: returns (node id, cosine score) over chunks and summaries of every level."""
        scores = self.embeddings @ self.embed([query])[0]
        node_ids_by_score = np.argsort(scores)[::-1]          # node ids sorted by score, best first
        top_ids = node_ids_by_score[:k]                       # keep the k best nodes
        return [(int(i), float(scores[i])) for i in top_ids]  # (node id, score) pairs

    def leaf_ids(self, node_id: int) -> list[int]:
        """All original chunks under a node (a leaf returns itself)."""
        node = self.nodes[node_id]
        if node.level == 0:
            return [node_id]

        leaves = set()
        for child in node.children:              # for each child of this summary
            for leaf in self.leaf_ids(child):    # take all the leaves under that child (recursive call)
                leaves.add(leaf)                 # add them; a set ignores duplicates
        return sorted(leaves)                    # back to a list, in corpus order

    def save(self, out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        # save tree as a json
        nodes_as_dicts = [asdict(n) for n in self.nodes]                       # 1. Node -> dict
        json_text = json.dumps(nodes_as_dicts, ensure_ascii=False, indent=2)   # 2. list of dicts -> JSON text
        path = out_dir / "nodes.json"                                          # 3. file path
        path.write_text(json_text, encoding="utf-8")                           # 4. write the text to the file

        # save embedding
        np.save(out_dir / "embeddings.npy", self.embeddings)

/lustre/fshomisc/sup/pub/miniforge/26.1.1-3/envs/pytorch-gpu-2.13.0+py3.12.13+cuda-13.2.1/lib/python3.12/site-packages/transformer_engine/jax/sharding.py:23: DeprecationWarning: jax.interpreters.pxla.thread_resources is deprecated as of JAX v0.8.2. Please switch to using `with jax.set_mesh(mesh)` instead of `with mesh:` and then use `jax.sharding.get_abstract_mesh()` to get the current mesh.
  _PXLA_THREAD_RESOURCES = pxla.thread_resources


In [5]:
raptor_summary = Raptor(
    embed_model="/linkhome/idris/genini01/ssos027/.cache/docling/models/all-MiniLM-L6-v2",
    llm_model="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3-VL-32B-Instruct"
)
raptor_summary.build(texts, cluster_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1058 [00:00<?, ?it/s]

[W1026 11:52:00.936406288 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 262144000 bytes (free: 11468800, total: 85019590656).
[W1026 11:52:00.942448366 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 262144000 bytes (free: 164560896, total: 85019590656).


OutOfMemoryError: CUDA out of memory. Tried to allocate 250.00 MiB. GPU 0 has a total capacity of 79.18 GiB of which 156.94 MiB is free. Including non-PyTorch memory, this process has 79.02 GiB memory in use. Of the allocated memory 77.80 GiB is allocated by PyTorch, and 313.25 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [14]:
print(raptor_summary.nodes[184].text)

Le logotype de l’Institut National des Jeunes Aveugles (INJA), fondé en 1785, s’adapte à tout type de situation et se décline sur fond clair ou sombre, avec des variantes incluant le logo, l’icône et l’image en page entière. Il est interdit d’utiliser le logo dans des contextes non conformes aux déclinaisons prévues. Les couleurs du logotype sont contrastées et énergiques, visant à dynamiser et moderniser la communication. Le noir et le gris sont privilégiés pour les fonds et aplats de couleurs, tandis que les couleurs désaturées peuvent être utilisées pour segmenter l’information, apportant noblesse et douceur à l’harmonie globale. Pour les supports imprimés, les couleurs ont été pensées pour le digital et doivent être adaptées en CMJN : gris clair (C:00


## JPG extraction

In [14]:
import torch
from pathlib import Path
from collections.abc import Iterable
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from docling_core.types.doc import (
    DoclingDocument, DocItemLabel, ImageRef, ImageRefMode,
    ProvenanceItem, BoundingBox, Size, CoordOrigin,
)


class Jpg2DoclingDocument:
    """Turn a jpg photo into a one-page DoclingDocument whose picture is captioned by a VLM.

    Why we build the DoclingDocument ourselves instead of using Docling's DocumentConverter:
    when we fed jpg files to Docling's image pipeline, it did not reliably detect the photo
    as a picture item, so there was no picture to caption or describe. Our working hypothesis
    is that its layout model is designed for document pages (text, tables, figures) and does
    not treat a whole photograph as a figure; we did not confirm this.

    Since the output must still be in the same DoclingDocument format as the other file types,
    we build it directly:
      - one page, sized like the image;
      - one picture covering that whole page (provenance = full-page bounding box);
      - one caption, generated by a VLM (transformers), linked to the picture.
    """

    DEFAULT_PROMPT = ("Décris cette photographie en français, en 2 à 4 phrases : "
                      "personnes, objets, lieu, ambiance. N'invente rien.")

    def __init__(
        self,
        vlm_model: str | Path,
        prompt: str | None = None,
        max_side: int = 1280,
        max_new_tokens: int = 200,
    ) -> None:
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = AutoProcessor.from_pretrained(vlm_model)
        self.model = AutoModelForImageTextToText.from_pretrained(vlm_model).to(self.device)
        self.prompt = prompt or self.DEFAULT_PROMPT
        self.max_side = max_side
        self.max_new_tokens = max_new_tokens

    def describe(self, img: Image.Image) -> str:
        input_img = img.copy()
        input_img.thumbnail((self.max_side, self.max_side))  # limit img size
        chat = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": self.prompt}]}]
        text = self.processor.apply_chat_template(chat, add_generation_prompt=True)
        inputs = self.processor(images=input_img, text=text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        return self.processor.decode(new_tokens, skip_special_tokens=True).strip()

    def convert(self, path: str | Path) -> DoclingDocument:
        path = Path(path)
        img = Image.open(path).convert("RGB")
        w, h = img.size

        doc = DoclingDocument(name=path.stem)
        doc.add_page(page_no=1, size=Size(width=w, height=h))  # empty page with img size
        caption = doc.add_text(label=DocItemLabel.CAPTION, text=self.describe(img))  # add caption
        doc.add_picture(
            image=ImageRef.from_pil(img, dpi=72),
            caption=caption,
            prov=ProvenanceItem(
                page_no=1, charspan=(0, 0),
                bbox=BoundingBox(l=0, t=0, r=w, b=h, coord_origin=CoordOrigin.TOPLEFT),
            ),
        )  # add img with reference to caption
        return doc

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for path in map(Path, paths):
            doc = self.convert(path)
            doc.save_as_json(out_dir / f"{path.stem}.json", image_mode=ImageRefMode.REFERENCED)
            doc.save_as_markdown(out_dir / f"{path.stem}.md", image_mode=ImageRefMode.REFERENCED)
            print("SAVED:", path.name)

In [15]:
jpg_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.jpg", case_sensitive=False))
img_converter = Jpg2DoclingDocument(vlm_model="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3-VL-8B-Instruct")
img_converter.convert_all(jpg_paths, "data/INJA/Livrable 1/converted/jpg2")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

SAVED: Logo_INJA_complet_clair_couleur.jpg
SAVED: Logo_INJA_complet_sombre_couleur.jpg
SAVED: Logo_INJA_icone_clair_couleur.jpg
SAVED: Logo_INJA_icone_sombre_couleur.jpg
SAVED: Logo_INJA_simple_clair_couleur.jpg
SAVED: Logo_INJA_simple_sombre_couleur.jpg
SAVED: 20230316_093719.jpg
SAVED: DSC_1717.JPG
SAVED: DSC_5262.JPG
SAVED: DSC_5862.JPG
SAVED: DSC_5872.JPG
SAVED: DSC_5976.JPG


## PPTX extraction

In [51]:
from collections.abc import Iterable
from pathlib import Path

from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.document import ConversionResult
from docling.document_converter import DocumentConverter


class SimplePptx2DoclingDocument:
    """Convert text-only pptx files to DoclingDocuments with Docling's default PPTX converter."""

    def __init__(self) -> None:
        self.converter = DocumentConverter(allowed_formats=[InputFormat.PPTX])

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            name = Path(result.input.file).stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            doc.save_as_json(out_dir / f"{name}.json")
            doc.save_as_markdown(out_dir / f"{name}.md")
            print("SAVED:", name, "|", len(doc.pages), "slides")

In [52]:
pptx_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.pptx"))
SimplePptx2DoclingDocument().convert_all(pptx_paths, "data/INJA/Livrable 1/converted/pptx")

SAVED: Présentation_INJA_24_04_04_vF | 10 slides


## XLSX extraction

In [1]:
import torch
import pandas as pd
from pathlib import Path
from collections.abc import Iterable
from transformers import AutoProcessor, AutoModelForImageTextToText
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.document_converter import DocumentConverter
from docling_core.types.doc import DocItemLabel


class Xlsx2DoclingDocument:
    """Convertit des fichiers Excel en DoclingDocument, avec une description de chaque tableau.

    Le tableau lui-même est extrait par le DocumentConverter de Docling (backend XLSX), et
    non reconstruit à la main. Ce choix est volontaire : Docling lit directement la structure
    du classeur (lignes, colonnes, cellules fusionnées, donc catégories de colonnes sur
    plusieurs niveaux) et produit les cellules du tableau. Le reconstruire nous-mêmes depuis
    pandas obligerait à gérer à la main les en-têtes multi-niveaux, les spans de colonnes et
    les types de valeurs, avec un risque d'erreur plus élevé.

    pandas n'est utilisé que pour lire les premières lignes brutes de chaque feuille
    (header=None, pas d'interprétation d'en-tête). Ces lignes sont données à un modèle de
    vision-langage (texte seul) qui produit une description en français, ajoutée au tableau
    comme légende (caption). Le tableau complet reste celui de Docling.
    """
    DEFAULT_PROMPT = (
        "Voici les premières lignes brutes d'une feuille Excel (« {sheet} »). "
        "Les premières lignes sont des en-têtes, parfois sur "
        "plusieurs niveaux : une catégorie de colonnes peut couvrir plusieurs colonnes, "
        "et n'apparaît alors qu'au-dessus de la première.\n\n"
        "{table}\n\n"
        "Décris en français, en 3 à 5 phrases : le sujet du tableau, les catégories de colonnes, "
        "le rôle de chaque colonne et ce que représente une ligne. "
        "N'invente rien, ne donne aucun total ni statistique."
    )

    def __init__(self, vlm_model: str | Path, n_head: int = 10, max_new_tokens: int = 400) -> None:
        self.converter = DocumentConverter(allowed_formats=[InputFormat.XLSX])
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = AutoProcessor.from_pretrained(vlm_model)
        self.model = AutoModelForImageTextToText.from_pretrained(vlm_model).to(self.device)
        self.n_head = n_head
        self.max_new_tokens = max_new_tokens

    def describe(self, head: pd.DataFrame, sheet: str) -> str:
        prompt = self.DEFAULT_PROMPT.format(
            sheet=sheet, table=head.fillna("").to_string(header=False, index=False))
        chat = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        text = self.processor.apply_chat_template(chat, add_generation_prompt=True)
        inputs = self.processor(text=text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        return self.processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            path = Path(result.input.file)
            name = path.stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            heads = pd.read_excel(path, sheet_name=None, header=None, nrows=self.n_head, dtype=object)
            for sheet_no, (sheet, head) in enumerate(heads.items(), start=1):
                table = next((t for t in doc.tables if t.prov and t.prov[0].page_no == sheet_no), None)
                if table is None:
                    continue
                caption = doc.add_text(label=DocItemLabel.CAPTION, text=self.describe(head, sheet))
                table.captions.append(caption.get_ref())

            doc.save_as_json(out_dir / f"{name}.json")
            doc.save_as_markdown(out_dir / f"{name}.md")
            print("SAVED:", name, "|", len(doc.tables), "tables")

In [12]:
xlsx_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.xlsx", case_sensitive=False))
xlsx_converter = Xlsx2DoclingDocument(vlm_model="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3-VL-8B-Instruct")
xlsx_converter.convert_all(xlsx_paths, "data/INJA/Livrable 1/converted/xlsx2")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

/lustre/fswork/projects/idris/sos/ssos027/synthesis_RAG/.local/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/lustre/fswork/projects/idris/sos/ssos027/synthesis_RAG/.local/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


SAVED: PNEAA-Retroplanning_Marche_Plan_Production-MaJ260331 | 6 tables
SAVED: Classif_formats_fichier_adaptation-IPTDV_DGCS | 5 tables
SAVED: Enquête_coûts_réponses_anonymisées | 1 tables
SAVED: Tableau_handicaps_edition_adaptee_detaille | 5 tables


## DOCX extraction

In [3]:
import contextlib
from pathlib import Path
from collections.abc import Iterable
from docling_core.types.doc import ImageRefMode
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.datamodel.pipeline_options import ConvertPipelineOptions, PictureDescriptionVlmOptions
from docling.document_converter import DocumentConverter, WordFormatOption


class Docx2DoclingDocument:
    """Convert Word (.docx) files to DoclingDocuments, with classified and described pictures.

    DOCX files go through Docling's automatic pipeline. Unlike a PDF, a DOCX already stores
    its structure (headings, paragraphs, lists, tables), so Docling's Word backend reads it
    directly through the simple pipeline: no layout model, no table structure model, no OCR.
    Rebuilding this ourselves would add complexity without improving the result.

    Pictures: the simple pipeline inherits the enrichment step of Docling's ConvertPipeline,
    so picture classification and picture description (VLM, same prompt style as for PDFs)
    run on the images embedded in the document.
    """

    DEFAULT_IMAGE_PROMPT = (
        "Tu analyses une image extraite d'un document professionnel en français. "
        "1) Indique d'abord le type d'image : schéma ou organigramme, graphique, tableau, logo, "
        "icône, capture d'écran, palette de couleurs, photographie ou autre. "
        "2) Décris ensuite précisément son contenu selon son type (éléments, liens, valeurs, "
        "légendes, couleurs). "
        "3) Recopie enfin tout le texte visible, tel quel, ou écris « Aucun texte visible ». "
        "N'invente rien."
    )

    def __init__(self,
                 artifacts_path: str | Path,
                 vlm_model_path: str | Path,
                 image_prompt: str | None = None,
                 num_threads: int = 4) -> None:
        # Docling loads the description model from "artifacts_path / repo_id":
        # link the local VLM there so the other Docling models stay in artifacts_path
        vlm_model_path = Path(vlm_model_path)
        link = Path(artifacts_path) / vlm_model_path.name
        if not link.exists():
            link.symlink_to(vlm_model_path, target_is_directory=True)

        opts = ConvertPipelineOptions(
            artifacts_path=artifacts_path,
            accelerator_options=AcceleratorOptions(num_threads=num_threads, device=AcceleratorDevice.AUTO),
            do_picture_classification=True,
            do_picture_description=True,
            picture_description_options=PictureDescriptionVlmOptions(
                repo_id=vlm_model_path.name,  # same name as the symlink
                prompt=image_prompt or self.DEFAULT_IMAGE_PROMPT,
            ),
        )
        self.converter = DocumentConverter(
            allowed_formats=[InputFormat.DOCX],
            format_options={InputFormat.DOCX: WordFormatOption(pipeline_options=opts)},
        )

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            name = Path(result.input.file).stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            with contextlib.chdir(out_dir):  # keeps the image folder next to the files, not nested
                doc.save_as_json(Path(f"{name}.json"), image_mode=ImageRefMode.REFERENCED)
                doc.save_as_markdown(Path(f"{name}.md"), image_mode=ImageRefMode.REFERENCED)
            print("SAVED:", name, "|", len(doc.tables), "tables |", len(doc.pictures), "pictures")

In [4]:
docx_paths = sorted(Path("data/INJA/Livrable 1/Donnée entrée").rglob("*.docx", case_sensitive=False))
docx_converter = Docx2DoclingDocument(artifacts_path="/linkhome/idris/genini01/ssos027/.cache/docling/models",
                                      vlm_model_path="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3-VL-8B-Instruct")
docx_converter.convert_all(docx_paths, "data/INJA/Livrable 1/converted/docx2")

NameError: name 'models_dir' is not defined

## VLM extraction

In [8]:
import contextlib
from pathlib import Path
from collections.abc import Iterable
from docling_core.types.doc import ImageRefMode
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.datamodel.pipeline_options import VlmPipelineOptions
from docling.datamodel.pipeline_options_vlm_model import (
    InferenceFramework, InlineVlmOptions, ResponseFormat, TransformersModelType,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline


class VlmPdf2DoclingDocument:
    """Convert PDFs to DoclingDocuments with Docling's VLM pipeline (one VLM reads each page image).

    A single vision-language model transcribes each rendered page (text, tables, images),
    and Docling turns its output (DocTags or Markdown) into a DoclingDocument.
    """

    # source: https://github.com/docling-project/docling/blob/main/docling/datamodel/vlm_model_specs.py#L274
    DEFAULT_PAGE_PROMPT = "Convert this page to markdown. Do not miss any text and only output the bare markdown!"

    def __init__(self,
                 artifacts_path: str | Path,
                 vlm_model_path: str | Path,
                 vlm_options: InlineVlmOptions | None = None,
                 num_threads: int = 4) -> None:
        # Docling loads the VLM from "artifacts_path / repo_id":
        # link the local VLM there so the other Docling models stay in artifacts_path
        vlm_model_path = Path(vlm_model_path)
        link = Path(artifacts_path) / vlm_model_path.name
        if not link.exists():
            link.symlink_to(vlm_model_path, target_is_directory=True)

        vlm_options = (vlm_options or InlineVlmOptions(
            repo_id="",  # set below
            prompt=self.DEFAULT_PAGE_PROMPT,
            response_format=ResponseFormat.MARKDOWN,
            inference_framework=InferenceFramework.TRANSFORMERS,
            transformers_model_type=TransformersModelType.AUTOMODEL_IMAGETEXTTOTEXT,
            max_new_tokens=4096,
        )).model_copy(update={"repo_id": vlm_model_path.name})  # same name as the symlink

        opts = VlmPipelineOptions(
            artifacts_path=artifacts_path,
            accelerator_options=AcceleratorOptions(num_threads=num_threads, device=AcceleratorDevice.AUTO),
            vlm_options=vlm_options,
            generate_page_images=True,  # the VLM reads page images
        )        self.converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_cls=VlmPipeline, pipeline_options=opts)},
        )

    def convert_all(self, paths: Iterable[str | Path], out_dir: str | Path) -> None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        for result in self.converter.convert_all(paths, raises_on_error=False):
            name = Path(result.input.file).stem.strip()

            if result.status == ConversionStatus.FAILURE:
                print("FAILED:", name, result.errors)
                continue

            doc = result.document
            with contextlib.chdir(out_dir):  # keeps the image folder next to the files, not nested
                doc.save_as_json(Path(f"{name}.json"), image_mode=ImageRefMode.REFERENCED)
                doc.save_as_markdown(Path(f"{name}.md"), image_mode=ImageRefMode.REFERENCED)
            print("SAVED:", name, "|", doc.num_pages(), "pages |", len(doc.tables), "tables |", len(doc.pictures), "pictures")

In [ ]:
pdf_paths = [
    "data/INJA/Livrable 1/Donnée entrée/Charte graphique INJA/CHARTE_GRAPHIQUE_INJA.pdf",
    "data/INJA/Livrable 1/Donnée entrée/DGCS commentaire relecture Version 1/20260331 Analyse livrable 1 par DGCS.pdf",
    "data/INJA/Livrable 1/Donnée entrée/Estimation couts/BP ArtSign Tradlitteraire .pdf",
    "data/INJA/Livrable 1/Donnée entrée/Organisme de médiation/Mission médiation.pdf",
]

pdf_converter = VlmPdf2DoclingDocument(artifacts_path="/linkhome/idris/genini01/ssos027/.cache/docling/models",
                                       vlm_model_path="/lustre/fsmisc/dataset/HuggingFace_Models/Qwen/Qwen3-VL-8B-Instruct")
pdf_converter.convert_all(pdf_paths, "data/INJA/Livrable 1/converted/pdf_vision")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

SAVED: CHARTE_GRAPHIQUE_INJA | 24 pages | 0 tables | 0 pictures
